# 01 · CFTR2 — the disease-specific *functional* truth set

[CFTR2](https://cftr2.org) classifies *CFTR* variants using **patient outcomes + in-vitro CFTR function assays** — a different, more *functional* kind of evidence than ClinVar's clinical assertions. That functional axis makes it an **orthogonal** truth set (tools/10). This notebook builds the full CFTR2 release locally from a manually-downloaded workbook (gitignored) — see the build cell below.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · CFTR2 — a disease-specific, *functional* reference

[CFTR2](https://cftr2.org) is different in kind from ClinVar. It is a **CF-specific** database that classifies *CFTR* variants as:

- **CF-causing**
- **Varying clinical consequence** (formerly "CF-causing (mild)")
- **Non CF-causing**
- **No interpretation available** — not yet enough evidence

Crucially, CFTR2's calls are built from **two kinds of evidence together**:
1. **Patient data** — real clinical outcomes across thousands of people with CF who carry the variant.
2. **In-vitro CFTR function assays** — measuring, in the lab, how much working chloride-channel the variant protein actually produces.

That second, functional axis makes CFTR2 **partially orthogonal** to ClinVar — but *not* independent of it (the two share clinical evidence and cross-cite; see tools/10).

### Building the data — a manual download (no API)

CFTR2 has no API — the variant list is published as an Excel workbook. **You must
fetch this one yourself:**

1. Go to <https://cftr2.org>, find the variant-list history / download page, and
   get the current release.
2. Save it into `data/` (gitignored — never commit it) and set `CFTR2_XLSX_NAME`
   below to match its filename.

**Version control.** CFTR2 has no historical archive to pin against the way
ClinVar does (checked directly against cftr2.org: there is no dated-release
listing, just the current file). What the cell below does instead:

- Reads the release date straight out of the workbook's own header metadata
  (the `Date:` row CFTR2 ships in every release) rather than assuming one.
- Persists that date into `data/cftr2_cftr.release.json` alongside the extract,
  so `load_cftr2()` can expose it as a `cftr2_release` column.
- If you want to reproduce a *past* run, you need to have manually saved that
  older workbook yourself (cftr2.org doesn't keep one for you) — point
  `CFTR2_XLSX_NAME` at it, and the recorded release date will reflect whatever
  that file's own header says, not today's date.

The cell reads two sheets from the workbook: **"CFTR2 variants by legacy
name"** (the variant list itself — legacy name, protein name, cDNA name, allele
count/frequency, and functional class) and **"Genomic coordinates"** (authoritative
GRCh38 positions). It derives the 1-letter `protein_variant` key for simple
single-residue missense variants only (regex on the protein name), resolves a
handful of variants listed under a **pipe-combined cDNA name** (e.g. W1282X is
`c.3845G>A|c.3846G>A`, two SNVs that create the same stop codon) by trying each
`|`-separated alternative against the genomic sheet, and writes
`data/cftr2_cftr.csv`. It also asserts the workbook's header states the
expected MANE transcript (`NM_000492.4`) before trusting its coordinates.

License: CFTR2's public data-use terms (cite CFTR2 if you use it) — see
`data_manifest.json`.

In [2]:
import re, json, openpyxl
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
# Change this if you've manually sourced a different (e.g. older) CFTR2 release --
# whatever release date IS in that file's own header is what gets recorded.
CFTR2_XLSX_NAME = "CFTR2_30January2026.xlsx"
CFTR2_XLSX = DATA_DIR / CFTR2_XLSX_NAME
CFTR2_TSV = DATA_DIR / "cftr2_cftr.csv"
CFTR2_RELEASE_JSON = DATA_DIR / "cftr2_cftr.release.json"
EXPECT_TX = "NM_000492.4"
AA3TO1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q",
    "Glu": "E", "Gly": "G", "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
    "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T", "Trp": "W",
    "Tyr": "Y", "Val": "V",
}
MIS = re.compile(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")   # simple single-residue missense only


def missense_key(protein_name: str) -> str:
    if not protein_name:
        return ""
    m = MIS.match(protein_name.strip())
    if not m:
        return ""
    a, pos, b = m.group(1), m.group(2), m.group(3)
    return f"{AA3TO1[a]}{pos}{AA3TO1[b]}" if a in AA3TO1 and b in AA3TO1 else ""


if CFTR2_TSV.exists():
    print(f"already built -> {CFTR2_TSV.name} (delete it and {CFTR2_RELEASE_JSON.name} to rebuild)")
elif not CFTR2_XLSX.exists():
    raise FileNotFoundError(
        f"{CFTR2_XLSX} not found.\n"
        "CFTR2 has no API -- get the variant-list workbook:\n"
        "  1. Go to https://cftr2.org and download the current release xlsx\n"
        f"  2. Save it as {CFTR2_XLSX} (do NOT commit it -- data/ is gitignored)\n"
        "     (or set CFTR2_XLSX_NAME above to whatever you saved it as)\n"
        "Then re-run this cell."
    )
else:
    wb = openpyxl.load_workbook(CFTR2_XLSX, read_only=True, data_only=True)
    ws = wb["CFTR2 variants by legacy name"]

    # Header rows (1-12) carry provenance: release date, official counts, and --
    # critically -- the reference transcript, so the build documents its own basis
    # instead of assuming GRCh38/MANE.
    header_meta = {}
    for r in ws.iter_rows(min_row=1, max_row=12, values_only=True):
        cell = str(r[0]).strip() if r[0] is not None else ""
        if ":" in cell:
            k, v = cell.split(":", 1)
            header_meta[k.strip()] = v.strip()
    tx = header_meta.get("CFTR reference transcript", "")
    assert EXPECT_TX in tx, (
        f"CFTR2 header transcript is {tx!r}, expected {EXPECT_TX}; the extract's "
        "genomic coordinates + MANE assumptions may no longer hold -- check the release.")
    print("CFTR2 header provenance:")
    for k in ("Date", "Number of patients in CFTR2", "Number of variants reported in CFTR2",
              "Number of variants with interpretations", "CFTR reference transcript"):
        if k in header_meta:
            print(f"  {k}: {header_meta[k]}")

    rows = []
    for r in ws.iter_rows(min_row=13, values_only=True):
        if r[0] is None:
            continue
        legacy, protein, cdna, alt, alleles, af, prev, cur, changed = r[:9]
        rows.append({"protein_variant": missense_key(protein or ""), "legacy_name": legacy,
                     "protein_name": protein, "cdna_name": cdna, "cftr2_alleles": alleles,
                     "cftr2_af": af, "cftr2_class": cur})
    df = pd.DataFrame(rows)

    # Merge GRCh38 genomic coordinates from sheet 2 (on cDNA name).
    ws2 = wb["Genomic coordinates"]
    g = pd.DataFrame(ws2.iter_rows(min_row=2, values_only=True),
                      columns=list(next(ws2.iter_rows(min_row=1, max_row=1, values_only=True))))
    gcols = {"Variant cDNA name": "cdna_name", "grch38_chr": "grch38_chr",
             "grch38_pos": "grch38_pos", "grch38_ref": "grch38_ref", "grch38_alt": "grch38_alt"}
    g = g[list(gcols)].rename(columns=gcols).drop_duplicates("cdna_name")
    gcoord = {k: v for k, v in g.set_index("cdna_name")
              [["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"]].to_dict("index").items()}

    def resolve_coords(cdna):
        # Some CFTR2 variants are listed under a PIPE-combined cDNA name (e.g. W1282X is
        # 'c.3845G>A|c.3846G>A') but the genomic sheet keys each single name separately.
        # Try the whole name, then each alternative, taking the first with coordinates.
        if not isinstance(cdna, str):
            return {}
        for alt in [cdna, *cdna.split("|")]:
            hit = gcoord.get(alt.strip())
            if hit and pd.notna(hit.get("grch38_pos")):
                return hit
        return {}

    coords = pd.DataFrame([resolve_coords(c) for c in df["cdna_name"]], index=df.index,
                           columns=["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"])
    df = pd.concat([df, coords], axis=1)
    df.to_csv(CFTR2_TSV, index=False)

    CFTR2_RELEASE_JSON.write_text(json.dumps({
        "release_date": header_meta.get("Date", "unknown"),
        "source_xlsx": CFTR2_XLSX_NAME,
        "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "row_count": len(df),
    }, indent=2))

    print(f"\nwrote {CFTR2_TSV.relative_to(DATA_DIR.parent)} and {CFTR2_RELEASE_JSON.name}  rows: {len(df):,}")
    print("with a 1-letter missense key:", (df["protein_variant"] != "").sum(),
          f"(of {len(df)}; the rest are non-missense -- join by cdna_name/genomic coords)")

already built -> cftr2_cftr.csv (delete it and cftr2_cftr.release.json to rebuild)


In [3]:
cftr2 = tk.load_cftr2()          # the built extract, once you've run the build cell above
print('source :', cftr2['source'].unique(), '| release:', cftr2['cftr2_release'].iloc[0])
print('variants:', len(cftr2), '| with a missense key:', (cftr2['protein_variant'] != '').sum())
print()
print(cftr2['cftr2_class'].value_counts().to_string())
cftr2.head(8)

source : ['REAL'] | release: 30 January 2026
variants: 2097 | with a missense key: 780

cftr2_class
CF-causing                      1245
No interpretation available      722
Varying clinical consequence      83
Non CF-causing                    42


,protein_variant,legacy_name,protein_name,cdna_name,cftr2_alleles,cftr2_af,cftr2_class,grch38_chr,grch38_pos,grch38_ref,grch38_alt,cftr2_release,source
0,,F508del,p.Phe508del,c.1521_1523del,137363,0.650682595473364,CF-causing,7.0,117559590.0,ATCT,A,30 January 2026,REAL
1,,G542X,p.Gly542X,c.1624G>T,5752,0.027246975453089916,CF-causing,7.0,117587778.0,G,T,30 January 2026,REAL
2,G551D,G551D,p.Gly551Asp,c.1652G>A,3831,0.01814728146049852,CF-causing,7.0,117587806.0,G,A,30 January 2026,REAL
3,N1303K,N1303K,p.Asn1303Lys,c.3909C>G,3551,0.01682093355944407,CF-causing,7.0,117652877.0,C,G,30 January 2026,REAL
4,,W1282X,p.Trp1282X,c.3845G>A|c.3846G>A,2500,0.011842391973700416,CF-causing,7.0,117642565.0,G,A,30 January 2026,REAL
5,R117H,R117H,p.Arg117His,c.350G>A,2262,0.010714996257804137,Varying clinical consequence,7.0,117530975.0,G,A,30 January 2026,REAL
6,,3849+10kbC->T,p.?,c.3718-2477C>T,1990,0.00942654401106553,CF-causing,7.0,117639961.0,C,T,30 January 2026,REAL
7,,621+1G->T,p.?,c.489+1G>T,1860,0.008810739628433109,CF-causing,7.0,117531115.0,G,T,30 January 2026,REAL


### Which variants get a `protein_variant` key — and what are the rest?

The 1-letter `protein_variant` key (e.g. `G551D`) is derived by the build cell above from
the protein name with a regex for *simple single-residue missense* only. It exists for
~780 of ~2,097 variants. **The other ~1,317 are NOT all splice variants** — most are
deletions and nonsense. They carry an empty key and must be joined by `cdna_name` or
genomic coordinates instead.

In [4]:
import re
cf = cftr2.copy()
cf["has_key"] = cf["protein_variant"].fillna("") != ""
print("with missense key:", int(cf["has_key"].sum()), "| without:", int((~cf["has_key"]).sum()))

def category(row):
    p, c = str(row.get("protein_name") or ""), str(row.get("cdna_name") or "")
    if "del" in p or "del" in c: return "deletion/indel"
    if "X" in p or "Ter" in p:   return "nonsense (stop-gain)"
    if "ins" in c or "dup" in c: return "insertion/dup"
    if ("+" in c) or ("-" in c and "c." in c): return "splice/intronic"
    if "=" in p: return "synonymous"
    return "other/complex"

nokey = cf[~cf["has_key"]]
print("\nWhat the NON-missense (no-key) variants actually are:")
print(nokey.apply(category, axis=1).value_counts().to_string())

with missense key: 780 | without: 1317

What the NON-missense (no-key) variants actually are:
deletion/indel          540
nonsense (stop-gain)    347
splice/intronic         291
other/complex            50
insertion/dup            48
synonymous               41


## 2 · How orthogonal is CFTR2, really?

CFTR2's **functional-assay** component (in-vitro chloride-channel measurements)
is a wet-lab signal no sequence model trained on — genuinely independent
evidence. But CFTR2 is **not** an independent gold standard: its
**patient/clinical** component overlaps the same evidence that feeds ClinVar,
**ClinVar entries cite CFTR2**, and **CFTR2 informs the ACMG CFTR guidance**
ClinVar submitters follow — the two databases cross-reference each other.

> **Rule of thumb:** use ClinVar for **breadth**; lean on CFTR2's *functional*
> measurements as **partial** orthogonal evidence — but never report "agrees
> with CFTR2" as if it were independent of ClinVar. See tools/10 for the full
> circularity/temporal-leakage argument this is a summary of.

## Key takeaways

1. **CFTR2** calls combine **patient data + functional assays**. The functional axis is *partially* orthogonal to ClinVar — but CFTR2 cross-cites ClinVar, so it is **not** an independent gold standard (tools/10).
2. ~2,097 variants once built locally; **780** have a 1-letter missense key (779 of which join to AlphaMissense). The other ~1,317 are non-missense (see the breakdown above) and join by `cdna_name` / genomic coordinates.
3. **Version:** the build cell reads the release date out of the workbook's own header and persists it as `cftr2_release` — CFTR2 has no historical archive to pin against, so reproducing a past run means manually sourcing that older workbook from cftr2.org yourself.
4. The **CFTR2-vs-ClinVar agreement** cross-check lived in the archived integration notebook.

**Next:** tools/07 — **SpliceAI**, the first splice predictor.